# End-to-End Time Series Forecasting System

## Objective
Forecast next 8 periods of sales for each state using historical sales data.

Models implemented:
- XGBoost
- ARIMA/SARIMA
- Prophet
- LSTM

Additional features:
- Feature engineering
- Model comparison
- API deployment with FastAPI

### 1. Data Loading
Load Excel dataset containing State, Date, Total Sales, and Category.

In [15]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings 
warnings.filterwarnings('ignore')

df = pd.read_excel("data/Forecasting Case- Study.xlsx")

print(df.shape)
df.head()

(8084, 4)


,State,Date,Total,Category
0,Alabama,2019-01-12 00:00:00,109574036.0,Beverages
1,Arizona,2019-01-12 00:00:00,109101594.6,Beverages
2,Arkansas,2019-01-12 00:00:00,58049432.2,Beverages
3,California,2019-01-12 00:00:00,444766890.6,Beverages
4,Colorado,2019-01-12 00:00:00,89816716.3,Beverages


### 2. Data Preprocessing

Steps:
- Clean column names
- Convert dates to datetime
- Convert Total to numeric
- Aggregate sales by State and Date

In [3]:
df.columns = df.columns.str.strip()

df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)

df["Total"] = (
    df["Total"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(float)
)

df = df.groupby(["State", "Date"])["Total"].sum().reset_index()

print(df.shape)
df.head()

(8084, 3)


,State,Date,Total
0,Alabama,2019-01-12,109574036.0
1,Alabama,2019-03-11,112189103.8
2,Alabama,2019-06-10,129106730.4
3,Alabama,2019-08-12,108083723.8
4,Alabama,2019-10-11,110932912.8


### 3. State-Level Time Series Creation

California is selected as sample state for modeling.

In [4]:
state = "California"

state_df = df[df["State"] == state].copy()
state_df = state_df.sort_values("Date")
state_df.set_index("Date", inplace=True)

state_df.head()

,State,Total
Date,,
2019-01-12,California,444766890.6
2019-03-11,California,492597099.4
2019-06-10,California,506587724.1
2019-08-12,California,449865324.4
2019-10-11,California,497063951.7


### 4. Feature Engineering

Created features:
- Lag variables
- Rolling mean
- Rolling standard deviation
- Month feature for seasonality

In [5]:
state_df["lag_1"] = state_df["Total"].shift(1)
state_df["lag_3"] = state_df["Total"].shift(3)
state_df["lag_6"] = state_df["Total"].shift(6)

state_df["rolling_mean_3"] = state_df["Total"].rolling(3).mean()
state_df["rolling_std_3"] = state_df["Total"].rolling(3).std()

state_df["month"] = state_df.index.month

state_df = state_df.dropna()

state_df.head()

,State,Total,lag_1,lag_3,lag_6,rolling_mean_3,rolling_std_3,month
Date,,,,,,,,
2019-10-20,California,491957816.8,504318965.7,449865324.4,444766890.6,4.977802e+08,6.211627e+06,10
2019-10-27,California,505768871.6,491957816.8,497063951.7,492597099.4,5.006819e+08,7.589966e+06,10
2019-11-17,California,486794358.5,505768871.6,504318965.7,506587724.1,4.948403e+08,9.810188e+06,11
2019-11-24,California,477240954.9,486794358.5,491957816.8,449865324.4,4.899347e+08,1.452091e+07,11
2019-12-15,California,461461505.5,477240954.9,505768871.6,497063951.7,4.751656e+08,1.279331e+07,12


### 5. Train Validation Split

Last 8 observations used as validation set to simulate future forecasting.

In [7]:
train = state_df.iloc[:-8]
val = state_df.iloc[-8:]

print("Train shape:", train.shape)
print("Validation shape:", val.shape)

Train shape: (174, 8)
Validation shape: (8, 8)


### 6. XGBoost Model Training
Train gradient boosting model using engineered lag features.

In [10]:
features = [col for col in train.columns if col not in ["Total", "State"]]

X_train = train[features]
y_train = train["Total"]

X_val = val[features]
y_val = val["Total"]

xgb = XGBRegressor()
xgb.fit(X_train, y_train)

xgb_preds = xgb.predict(X_val)

xgb_mae = mean_absolute_error(y_val, xgb_preds)

print("XGBoost MAE:", round(xgb_mae, 2))

XGBoost MAE: 43353980.0


##### XGBoost Result

Validation Metric:
- MAE = 43,353,980

Interpretation:
The XGBoost model achieves an average absolute forecasting error of approximately 43.3 million sales units on the validation set.

### 7. ARIMA Model Training
Classical time series forecasting model for trend and seasonality.

In [13]:
arima = SARIMAX(
    train["Total"],
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)

arima_fit = arima.fit(disp=False)

arima_preds = arima_fit.forecast(steps=8)

arima_mae = mean_absolute_error(y_val, arima_preds)

print("ARIMA MAE:", round(arima_mae, 2))

ARIMA MAE: 43040681.97


##### ARIMA Result

Validation Metric:
- MAE = 43,040,681.97

Interpretation:
ARIMA slightly outperformed XGBoost on the validation set, indicating strong performance for this dataset despite irregular time intervals.

### 8. Prophet Model Training
Forecasting using Prophet to capture trend and seasonality.

In [14]:
prophet_train = train.reset_index()[["Date", "Total"]]
prophet_train.columns = ["ds", "y"]

model = Prophet()
model.fit(prophet_train)

future = model.make_future_dataframe(periods=8, freq="W")
forecast = model.predict(future)

prophet_preds = forecast["yhat"].tail(8).values

prophet_mae = mean_absolute_error(y_val, prophet_preds)

print("Prophet MAE:", round(prophet_mae, 2))

16:09:35 - cmdstanpy - INFO - Chain [1] start processing
16:09:36 - cmdstanpy - INFO - Chain [1] done processing


Prophet MAE: 38924986.2


##### Prophet Result

Validation Metric:
- MAE = 38,924,986.20

Interpretation:
Prophet achieved the lowest validation error among all tested models and was selected as the final forecasting model.

### 9. LSTM Model Training

Deep learning model used to capture sequential dependencies in time series data.

In [16]:
series = train["Total"].values

In [17]:
X = []
y = []

window = 3

for i in range(window, len(series)):
    X.append(series[i-window:i])
    y.append(series[i])

X = np.array(X)
y = np.array(y)

X = X.reshape((X.shape[0], X.shape[1], 1))

print(X.shape)
print(y.shape)

(171, 3, 1)
(171,)


In [18]:
lstm = Sequential([
    LSTM(50, input_shape=(window, 1)),
    Dense(1)
])

lstm.compile(optimizer="adam", loss="mse")

lstm.fit(X, y, epochs=10, verbose=0)

In [19]:
full_series = state_df["Total"].values

X_test = []

for i in range(len(full_series)-8, len(full_series)):
    X_test.append(full_series[i-window:i])

X_test = np.array(X_test)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

In [20]:
lstm_preds = lstm.predict(X_test).flatten()

lstm_mae = mean_absolute_error(y_val, lstm_preds)

print("LSTM MAE:", round(lstm_mae, 2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 451ms/step
LSTM MAE: 846171878.58


##### LSTM Result

Validation Metric:
- MAE = 846,171,878.58

Interpretation:
LSTM underperformed compared to classical and machine learning models, likely due to limited dataset size and irregular time intervals.

In [21]:
scores = {
    "XGBoost": xgb_mae,
    "ARIMA": arima_mae,
    "Prophet": prophet_mae,
    "LSTM": lstm_mae
}

best_model = min(scores, key=scores.get)

print("Scores:")
print(scores)
print("\nBest Model:", best_model)

Scores:
{'XGBoost': 43353979.99999997, 'ARIMA': 43040681.96629949, 'Prophet': 38924986.20000397, 'LSTM': 846171878.5785824}

Best Model: Prophet


In [22]:
results = pd.DataFrame({
    "Model": ["XGBoost", "ARIMA", "Prophet", "LSTM"],
    "MAE": [xgb_mae, arima_mae, prophet_mae, lstm_mae]
})

results = results.sort_values("MAE")
results

,Model,MAE
2,Prophet,3.892499e+07
1,ARIMA,4.304068e+07
0,XGBoost,4.335398e+07
3,LSTM,8.461719e+08


### 10. Final Forecast Generation

Prophet was selected as the final model based on lowest validation MAE.
Next 8 periods are forecasted below.

In [23]:
future = model.make_future_dataframe(periods=8, freq="W")
final_forecast = model.predict(future)

final_forecast[["ds", "yhat"]].tail(8)

,ds,yhat
174,2023-04-30,8.733816e+08
175,2023-05-07,8.698508e+08
176,2023-05-14,8.683765e+08
177,2023-05-21,8.703588e+08
178,2023-05-28,8.692511e+08
179,2023-06-04,8.629538e+08
180,2023-06-11,8.586364e+08
181,2023-06-18,8.639162e+08


### 11. Forecasting for All States
Generate forecasts for each state using the selected best model.

In [26]:
all_forecasts = {}

states = df["State"].unique()

for state in states:
    try:
        state_df = df[df["State"] == state].copy()

        state_df["Date"] = pd.to_datetime(state_df["Date"])

        state_df = state_df.sort_values("Date")
        state_df.set_index("Date", inplace=True)

        state_df["lag_1"] = state_df["Total"].shift(1)
        state_df["lag_3"] = state_df["Total"].shift(3)
        state_df["lag_6"] = state_df["Total"].shift(6)

        state_df["rolling_mean_3"] = state_df["Total"].rolling(3).mean()
        state_df["rolling_std_3"] = state_df["Total"].rolling(3).std()

        state_df["month"] = state_df.index.month
        state_df = state_df.dropna()

        prophet_train = state_df.reset_index()[["Date", "Total"]]
        prophet_train.columns = ["ds", "y"]

        prophet_train["ds"] = pd.to_datetime(prophet_train["ds"])

        model = Prophet()
        model.fit(prophet_train)

        future = model.make_future_dataframe(periods=8, freq="W")
        forecast = model.predict(future)

        preds = forecast["yhat"].tail(8).tolist()

        all_forecasts[state] = preds

    except Exception as e:
        print(f"Skipping {state}: {e}")

17:03:39 - cmdstanpy - INFO - Chain [1] start processing
17:03:39 - cmdstanpy - INFO - Chain [1] done processing
17:03:39 - cmdstanpy - INFO - Chain [1] start processing
17:03:39 - cmdstanpy - INFO - Chain [1] done processing
17:03:40 - cmdstanpy - INFO - Chain [1] start processing
17:03:40 - cmdstanpy - INFO - Chain [1] done processing
17:03:41 - cmdstanpy - INFO - Chain [1] start processing
17:03:41 - cmdstanpy - INFO - Chain [1] done processing
17:03:42 - cmdstanpy - INFO - Chain [1] start processing
17:03:42 - cmdstanpy - INFO - Chain [1] done processing
17:03:43 - cmdstanpy - INFO - Chain [1] start processing
17:03:43 - cmdstanpy - INFO - Chain [1] done processing
17:03:44 - cmdstanpy - INFO - Chain [1] start processing
17:03:44 - cmdstanpy - INFO - Chain [1] done processing
17:03:44 - cmdstanpy - INFO - Chain [1] start processing
17:03:44 - cmdstanpy - INFO - Chain [1] done processing
17:03:45 - cmdstanpy - INFO - Chain [1] start processing
17:03:45 - cmdstanpy - INFO - Chain [1]

In [30]:
list(all_forecasts.items())

[('Alabama',
  [196174995.30646497,
   198369402.580962,
   203685160.6465513,
   205371505.0870373,
   200816813.70816967,
   194808994.04916227,
   193543248.42749995,
   197181024.25452176]),
 ('Arizona',
  [211667340.25836933,
   212458263.4381049,
   216673064.10917795,
   218517627.9736467,
   215343856.66795155,
   210974009.1057904,
   210686804.19138318,
   214474892.61853763]),
 ('Arkansas',
  [106058646.79155058,
   107584305.43243673,
   110171872.70251122,
   110371398.27547732,
   107483895.92280075,
   104485995.29998599,
   104367706.55125165,
   106552590.14595808]),
 ('California',
  [824876028.8718061,
   826496865.3726209,
   835809702.6283923,
   836912816.9643312,
   824636498.5599242,
   811424005.4591763,
   810962570.9223548,
   820443771.1879686]),
 ('Colorado',
  [172959448.03476492,
   175530824.69271603,
   176934761.65544122,
   174717991.84936807,
   170413691.07704157,
   168071710.9738221,
   169456282.38925862,
   171944722.99787486]),
 ('Connecticut',

##### Multi-State Forecasting

The final selected model (Prophet) was applied across all unique states to generate next 8 period forecasts.

Sample forecasts are shown below for selected states.

The final Prophet pipeline is generalized to forecast all states in the dataset.

In [29]:
"Colorado" in all_forecasts

True